# Notebook 04: Inpainting Comparison

**Goal:** Compare text removal methods. Given OCR bounding boxes, erase original text from the image while preserving the background.

**Methods evaluated:**
| # | Method | Type | Strengths |
|---|--------|------|-----------|
| 1 | **OpenCV INPAINT_TELEA** | Classical CV | Fast, no GPU needed |
| 2 | **OpenCV INPAINT_NS** | Classical CV | Better for gradients |
| 3 | **LaMa** | Deep learning | Excellent quality on complex backgrounds |
| 4 | **Simple Background Fill** | Naive | Trivial, works on uniform backgrounds |

**Input:** Page images from `data/page_images/` + OCR bboxes from `data/ocr_results/best/`  
**Output:** Inpainted images saved in `data/inpainted/<method>/`

In [ ]:
# Install dependencies (run once)
# !pip install opencv-python-headless Pillow numpy matplotlib
# For LaMa: pip install torch torchvision (+ download LaMa model weights)

In [ ]:
import json
import sys
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    DATA_DIR, PAGE_IMAGES_DIR, OCR_RESULTS_DIR, INPAINTED_DIR,
    load_json, save_json, load_image, save_image,
    draw_bboxes, display_images, display_comparison,
    pil_to_cv2, cv2_to_pil, sample_background_color,
)

# Load inputs
PAGE_INDEX = 0
page_image = load_image(PAGE_IMAGES_DIR / f"page_{PAGE_INDEX}.png")
ocr_blocks = load_json(OCR_RESULTS_DIR / "best" / f"page_{PAGE_INDEX}.json")

print(f"Loaded page {PAGE_INDEX}: {page_image.size[0]}x{page_image.size[1]}")
print(f"Loaded {len(ocr_blocks)} OCR text blocks")

## Create Inpainting Mask

Build a binary mask where white pixels = text regions to remove. The mask is dilated slightly to catch text edges and anti-aliasing.

In [ ]:
def create_text_mask(
    image_size: tuple[int, int],
    text_blocks: list[dict],
    dilation_px: int = 3,
    padding: int = 2,
) -> np.ndarray:
    """
    Create a binary mask for text regions.
    
    Args:
        image_size: (width, height) of the image
        text_blocks: OCR text blocks with 'bbox' field
        dilation_px: Pixels to dilate the mask (catches text edges)
        padding: Extra padding around each bbox
    
    Returns:
        Binary mask (uint8): 255 = text region, 0 = keep
    """
    w, h = image_size
    mask = np.zeros((h, w), dtype=np.uint8)
    
    for block in text_blocks:
        x0, y0, x1, y1 = block["bbox"]
        # Add padding
        x0 = max(0, int(x0) - padding)
        y0 = max(0, int(y0) - padding)
        x1 = min(w, int(x1) + padding)
        y1 = min(h, int(y1) + padding)
        mask[y0:y1, x0:x1] = 255
    
    # Dilate to catch text edges
    if dilation_px > 0:
        kernel = np.ones((dilation_px * 2 + 1, dilation_px * 2 + 1), np.uint8)
        mask = cv2.dilate(mask, kernel, iterations=1)
    
    return mask


# Create the mask
mask = create_text_mask(page_image.size, ocr_blocks, dilation_px=3, padding=2)

# Display original image with mask overlay
mask_overlay = page_image.copy()
mask_rgb = np.array(mask_overlay)
mask_rgb[mask > 0] = [255, 0, 0]  # Red overlay on text regions
mask_overlay = Image.fromarray(mask_rgb)

display_images(
    [page_image, Image.fromarray(mask), mask_overlay],
    titles=["Original", "Text Mask", "Mask Overlay"],
    cols=3,
    figsize=(20, 8),
)

---
## Option 1: OpenCV INPAINT_TELEA

Fast Marching Method by Alexandru Telea. Good for text removal on relatively uniform backgrounds.

In [ ]:
def inpaint_opencv(image: Image.Image, mask: np.ndarray, method: str = "telea", radius: int = 7) -> Image.Image:
    """
    Remove text using OpenCV inpainting.
    
    Args:
        image: Input PIL Image
        mask: Binary mask (255 = regions to inpaint)
        method: "telea" or "ns" (Navier-Stokes)
        radius: Inpainting radius (larger = smoother but slower)
    """
    img_cv = pil_to_cv2(image)
    
    flag = cv2.INPAINT_TELEA if method == "telea" else cv2.INPAINT_NS
    result = cv2.inpaint(img_cv, mask, inpaintRadius=radius, flags=flag)
    
    return cv2_to_pil(result)


# Method 1: TELEA
t0 = time.time()
telea_result = inpaint_opencv(page_image, mask, method="telea", radius=7)
telea_time = time.time() - t0

# Save
telea_dir = INPAINTED_DIR / "opencv_telea"
telea_dir.mkdir(parents=True, exist_ok=True)
save_image(telea_result, telea_dir / f"page_{PAGE_INDEX}.png")

print(f"OpenCV TELEA: {telea_time:.2f}s")
display_comparison(page_image, telea_result, "Original", "TELEA Inpainted")

---
## Option 2: OpenCV INPAINT_NS

Navier-Stokes based method. Often better for gradients and complex textures, but slower.

In [ ]:
# Method 2: Navier-Stokes
t0 = time.time()
ns_result = inpaint_opencv(page_image, mask, method="ns", radius=7)
ns_time = time.time() - t0

# Save
ns_dir = INPAINTED_DIR / "opencv_ns"
ns_dir.mkdir(parents=True, exist_ok=True)
save_image(ns_result, ns_dir / f"page_{PAGE_INDEX}.png")

print(f"OpenCV NS: {ns_time:.2f}s")
display_comparison(page_image, ns_result, "Original", "Navier-Stokes Inpainted")

---
## Option 3: LaMa (Large Mask Inpainting)

Deep learning inpainter from Samsung Research. Excellent quality, especially for large masked regions. Requires PyTorch.

**Setup:** `pip install torch torchvision` + download LaMa checkpoint.

In [ ]:
def inpaint_lama(image: Image.Image, mask: np.ndarray) -> Image.Image:
    """
    Remove text using LaMa (Large Mask Inpainting) model.
    
    Uses the simple_lama package for easy inference.
    Install: pip install simple-lama-inpainting
    Or use the full LaMa repo: https://github.com/advimman/lama
    
    Requires: pip install torch torchvision simple-lama-inpainting
    """
    from simple_lama_inpainting import SimpleLama
    
    lama = SimpleLama()
    
    # SimpleLama expects PIL Image and PIL mask
    mask_pil = Image.fromarray(mask).convert("L")
    result = lama(image, mask_pil)
    
    return result


# Run LaMa
try:
    t0 = time.time()
    lama_result = inpaint_lama(page_image, mask)
    lama_time = time.time() - t0
    
    # Save
    lama_dir = INPAINTED_DIR / "lama"
    lama_dir.mkdir(parents=True, exist_ok=True)
    save_image(lama_result, lama_dir / f"page_{PAGE_INDEX}.png")
    
    print(f"LaMa: {lama_time:.2f}s")
    display_comparison(page_image, lama_result, "Original", "LaMa Inpainted")
except Exception as e:
    print(f"LaMa skipped: {e}")
    print("Install: pip install torch torchvision simple-lama-inpainting")
    lama_result = None
    lama_time = None

---
## Option 4: Simple Background Fill

Naive approach: fill each text bbox with the sampled background color. Only works well on uniform backgrounds (white pages, solid-color boxes).

In [ ]:
def inpaint_simple_fill(image: Image.Image, text_blocks: list[dict], padding: int = 2) -> Image.Image:
    """
    Remove text by filling each bbox with the sampled background color.
    
    Samples pixels around each bbox to determine the background color,
    then fills the entire bbox with that color.
    """
    from PIL import ImageDraw
    
    result = image.copy()
    draw = ImageDraw.Draw(result)
    
    for block in text_blocks:
        bbox = block["bbox"]
        x0, y0, x1, y1 = bbox
        x0 = max(0, int(x0) - padding)
        y0 = max(0, int(y0) - padding)
        x1 = min(image.size[0], int(x1) + padding)
        y1 = min(image.size[1], int(y1) + padding)
        
        bg_color = sample_background_color(image, [x0, y0, x1, y1])
        draw.rectangle([x0, y0, x1, y1], fill=bg_color)
    
    return result


# Method 4: Simple fill
t0 = time.time()
fill_result = inpaint_simple_fill(page_image, ocr_blocks)
fill_time = time.time() - t0

# Save
fill_dir = INPAINTED_DIR / "simple_fill"
fill_dir.mkdir(parents=True, exist_ok=True)
save_image(fill_result, fill_dir / f"page_{PAGE_INDEX}.png")

print(f"Simple Fill: {fill_time:.2f}s")
display_comparison(page_image, fill_result, "Original", "Simple Fill")

---
## Side-by-Side Comparison of All Methods

In [ ]:
# Collect all available results
all_inpainted = {"Original": page_image}
all_inp_times = {}

all_inpainted["TELEA"] = telea_result
all_inp_times["TELEA"] = telea_time

all_inpainted["Navier-Stokes"] = ns_result
all_inp_times["Navier-Stokes"] = ns_time

if lama_result:
    all_inpainted["LaMa"] = lama_result
    all_inp_times["LaMa"] = lama_time

all_inpainted["Simple Fill"] = fill_result
all_inp_times["Simple Fill"] = fill_time

# Display all
titles = []
for name in all_inpainted:
    if name == "Original":
        titles.append("Original")
    else:
        titles.append(f"{name}\n({all_inp_times[name]:.2f}s)")

display_images(
    list(all_inpainted.values()),
    titles=titles,
    cols=min(len(all_inpainted), 3),
    figsize=(20, 14),
)

## Select Best Inpainting Method

In [ ]:
# Auto-detect which inpainting methods produced results
METHOD_DIRS = {
    "opencv_telea": "opencv_telea",
    "opencv_ns": "opencv_ns",
    "lama": "lama",
    "simple_fill": "simple_fill",
}

available_methods = {}
for name, dirname in METHOD_DIRS.items():
    method_dir = INPAINTED_DIR / dirname
    if method_dir.exists() and any(method_dir.iterdir()):
        available_methods[name] = dirname

if not available_methods:
    raise RuntimeError("No inpainting method produced results! Run at least one method cell above.")

print(f"Available methods: {list(available_methods.keys())}")

# ── SELECT YOUR BEST METHOD (set to None for auto-select, or pick one) ────
BEST_INPAINTING = None  # e.g., "opencv_telea", "opencv_ns", "lama", "simple_fill"
# ──────────────────────────────────────────────────────────────────────────────

if BEST_INPAINTING is None:
    BEST_INPAINTING = list(available_methods.keys())[0]
    print(f"Auto-selected: {BEST_INPAINTING}")

if BEST_INPAINTING not in available_methods:
    raise ValueError(f"'{BEST_INPAINTING}' has no results. Choose from: {list(available_methods.keys())}")

import shutil

source_dir = INPAINTED_DIR / available_methods[BEST_INPAINTING]
best_dir = INPAINTED_DIR / "best"

if best_dir.exists():
    shutil.rmtree(best_dir)
best_dir.mkdir(parents=True, exist_ok=True)

for f in source_dir.iterdir():
    shutil.copy2(f, best_dir / f.name)

print(f"Copied {BEST_INPAINTING} results to data/inpainted/best/")
print(f"\n✓ Inpainting selection complete. Next notebook: 05_text_overlay.ipynb")